In [1]:
#/kaggle/working/vilt_fakeddit_metadata_3way_fold_best.csv
"""
ViLT + Fakeddit — **3-way classification only** (metadata text prefix preserved).

Label mapping from the dataset’s `6_way_label` field (used only to build targets,
not as a 6-class head and not as binary):
  - Class 0 **true**: 6_way_label == 0
  - Class 1 **satire**: 6_way_label == 1
  - Class 2 **misinformation**: 6_way_label in {2, 3, 4, 5}
    (False connection, Imposter, Manipulated, Misleading — merged)

Metrics: accuracy + **macro** precision / recall / F1 over the 3 classes.

Same training layout as the 2-way metadata script: 80/10/10 split, 5-fold on train pool,
5 epochs, CSV + printed tables. Includes GPU capability fallback (P100 → CPU).

Run: python vilt_fakeddit_kaggle_80_10_10_ep5_metadata_3way.py
"""

from __future__ import annotations

import csv
import io
import os
import random
from collections.abc import Callable
from contextlib import nullcontext
from dataclasses import dataclass
from typing import Any
from urllib.parse import urlparse

import numpy as np
import requests
import torch
import torch.nn as nn
from PIL import Image
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.model_selection import StratifiedKFold, train_test_split
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from transformers import AutoConfig, ViltModel, ViltProcessor, get_linear_schedule_with_warmup

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Human-readable names for logs (model never sees these strings as labels)
THREE_WAY_NAMES = ("true", "satire", "misinformation")


@dataclass
class Config:
    model_name: str = "dandelin/vilt-b32-mlm"
    dataset_name: str = "AdoCleanCode/Fakeddit"
    max_samples: int = 5000
    num_classes: int = 3
    batch_size: int = 8
    epochs: int = 5
    lr: float = 2e-5
    weight_decay: float = 0.01
    grad_accumulation: int = 1
    n_splits: int = 5
    num_workers: int = 0
    image_timeout_s: float = 10.0
    image_size: int = 384
    train_frac: float = 0.8
    val_frac: float = 0.1
    test_frac: float = 0.1
    use_metadata_prefix: bool = True
    use_split_flags: bool = True
    results_dir: str = "."
    results_file_prefix: str = "vilt_fakeddit_metadata_3way"
    force_cpu: bool = False


config = Config()


def six_way_to_three_way(six: int) -> int:
    """Map Fakeddit 6-way taxonomy to 3 training classes (not 2-way, not 6-way head)."""
    if six == 0:
        return 0
    if six == 1:
        return 1
    if six in (2, 3, 4, 5):
        return 2
    raise ValueError(f"unexpected 6_way_label: {six}")


def resolve_training_device(*, force_cpu: bool = False) -> torch.device:
    if force_cpu:
        print("force_cpu=True: using CPU.")
        return torch.device("cpu")
    if not torch.cuda.is_available():
        return torch.device("cpu")
    try:
        major, minor = torch.cuda.get_device_capability(0)
        name = torch.cuda.get_device_name(0)
    except Exception as e:
        print(f"Could not query CUDA device ({e}); using CPU.")
        return torch.device("cpu")
    if major < 7:
        print(
            f"WARNING: GPU '{name}' is compute {major}.{minor} (sm_{major}{minor}); "
            "using CPU (PyTorch wheel likely sm_70+ only). Use T4+ on Kaggle for CUDA."
        )
        return torch.device("cpu")
    try:
        t = torch.zeros(1, device="cuda")
        _ = (t + 1.0).item()
        del t
        torch.cuda.synchronize()
    except Exception as e:
        print(f"WARNING: CUDA probe failed ({type(e).__name__}: {e}); using CPU.")
        return torch.device("cpu")
    return torch.device("cuda")


def build_amp_helpers(use_cuda: bool) -> tuple[Callable[[], Any], Any]:
    if not use_cuda:
        return (lambda: nullcontext()), torch.cuda.amp.GradScaler(enabled=False)
    try:
        scaler = torch.amp.GradScaler("cuda", enabled=True)
        return (lambda: torch.amp.autocast("cuda"), scaler)
    except Exception:
        scaler = torch.cuda.amp.GradScaler(enabled=True)
        return (lambda: torch.cuda.amp.autocast(), scaler)


def load_pil_from_url(url: str, timeout: float) -> Image.Image | None:
    try:
        r = requests.get(url, timeout=timeout, stream=True)
        r.raise_for_status()
        return Image.open(io.BytesIO(r.content)).convert("RGB")
    except Exception:
        return None


def image_url_host(image_url: str | None) -> str:
    if not image_url or str(image_url).lower() == "null":
        return "?"
    try:
        host = urlparse(str(image_url).strip()).netloc.lower()
        return (host or "?")[:32]
    except Exception:
        return "?"


def build_metadata_prefix(ex: dict, body_text: str) -> str:
    if not config.use_metadata_prefix:
        return ""
    parts: list[str] = [f"h={image_url_host(ex.get('image_url'))}"]
    pid = ex.get("id")
    if pid is not None:
        parts.append(f"id={str(pid)[:10]}")
    parts.append(f"nch={len(body_text)}")
    if config.use_split_flags:
        gt = ex.get("general_train")
        if gt is not None:
            parts.append(f"gentr={'1' if gt else '0'}")
        gte = ex.get("general_test")
        if gte is not None:
            parts.append(f"gente={'1' if gte else '0'}")
    return "[" + "|".join(parts) + "] "


class FakedditThreeWayDataset(Dataset):
    def __init__(self, hf_dataset, processor: ViltProcessor, indices: list[int] | None = None):
        self.rows = hf_dataset
        self.processor = processor
        self.indices = indices if indices is not None else list(range(len(hf_dataset)))
        self.text_max_length = min(getattr(processor.tokenizer, "model_max_length", 40) or 40, 40)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx: int):
        i = self.indices[idx]
        ex = self.rows[i]
        body = ex.get("text") or ""
        text = build_metadata_prefix(ex, body) + body
        url = ex.get("image_url")
        if not url or str(url).lower() == "null":
            raise ValueError("missing image_url")
        six = int(ex["6_way_label"])
        label = six_way_to_three_way(six)

        img = load_pil_from_url(str(url), config.image_timeout_s)
        if img is None:
            img = Image.new("RGB", (config.image_size, config.image_size), color=(0, 0, 0))
        else:
            s = config.image_size
            if img.size != (s, s):
                img = img.resize((s, s), Image.BICUBIC)

        enc = self.processor(
            text=text,
            images=img,
            return_tensors="pt",
            padding="max_length",
            truncation=True,
            max_length=self.text_max_length,
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item["labels"] = torch.tensor(label, dtype=torch.long)
        return item


def collate_fn(batch):
    labels = torch.stack([b.pop("labels") for b in batch], dim=0)
    keys = batch[0].keys()
    out = {k: torch.stack([b[k] for b in batch], dim=0) for k in keys}
    out["labels"] = labels
    return out


class ViltThreeWayDetector(nn.Module):
    def __init__(self, num_classes: int):
        super().__init__()
        self.vilt = ViltModel.from_pretrained(config.model_name)
        hid = self.vilt.config.hidden_size
        self.dropout = nn.Dropout(self.vilt.config.hidden_dropout_prob)
        self.classifier = nn.Linear(hid, num_classes)

    def forward(
        self,
        input_ids: torch.Tensor,
        attention_mask: torch.Tensor,
        pixel_values: torch.Tensor,
        labels: torch.Tensor | None = None,
        token_type_ids: torch.Tensor | None = None,
        pixel_mask: torch.Tensor | None = None,
        **unused_kwargs,
    ):
        outputs = self.vilt(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
            pixel_values=pixel_values,
            pixel_mask=pixel_mask,
            return_dict=True,
        )
        pooled = outputs.pooler_output
        logits = self.classifier(self.dropout(pooled))
        loss = None
        if labels is not None:
            loss = nn.functional.cross_entropy(logits, labels)
        return {"loss": loss, "logits": logits}


def build_valid_indices(hf_dataset, max_n: int) -> list[int]:
    valid = []
    n = min(len(hf_dataset), max_n)
    for i in tqdm(range(n), desc="Scanning samples"):
        ex = hf_dataset[i]
        url = ex.get("image_url")
        if not url or str(url).lower() == "null":
            continue
        if ex.get("hasImage") is False:
            continue
        six = ex.get("6_way_label")
        if six is None:
            continue
        try:
            six_way_to_three_way(int(six))
        except ValueError:
            continue
        valid.append(i)
    return valid


def collect_predictions(
    model, data_loader, device, autocast_ctx: Callable[[], Any]
) -> tuple[np.ndarray, np.ndarray]:
    model.eval()
    all_preds, all_y = [], []
    with torch.no_grad():
        for batch in data_loader:
            labels = batch.pop("labels").to(device)
            batch = {k: v.to(device) for k, v in batch.items()}
            with autocast_ctx():
                logits = model(**batch, labels=None)["logits"]
            all_preds.append(logits.argmax(dim=-1).cpu().numpy())
            all_y.append(labels.cpu().numpy())
    return np.concatenate(all_y), np.concatenate(all_preds)


def multiclass_macro_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict[str, float]:
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, average="macro", zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, average="macro", zero_division=0)),
        "f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
    }


_PER_EPOCH_FIELDS = [
    "fold",
    "epoch",
    "val_accuracy",
    "val_precision_macro",
    "val_recall_macro",
    "val_f1_macro",
    "test_accuracy",
    "test_precision_macro",
    "test_recall_macro",
    "test_f1_macro",
]


def _flat_metrics_row(fold: int, epoch: int, m_val: dict, m_test: dict) -> dict[str, float | int | str]:
    return {
        "fold": fold,
        "epoch": epoch,
        "val_accuracy": m_val["accuracy"],
        "val_precision_macro": m_val["precision"],
        "val_recall_macro": m_val["recall"],
        "val_f1_macro": m_val["f1"],
        "test_accuracy": m_test["accuracy"],
        "test_precision_macro": m_test["precision"],
        "test_recall_macro": m_test["recall"],
        "test_f1_macro": m_test["f1"],
    }


def save_and_print_results_tables(fold_reports: list[dict]) -> tuple[str, str]:
    os.makedirs(config.results_dir, exist_ok=True)
    prefix = config.results_file_prefix
    path_epoch = os.path.join(config.results_dir, f"{prefix}_per_epoch.csv")
    path_best = os.path.join(config.results_dir, f"{prefix}_fold_best.csv")

    with open(path_epoch, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=_PER_EPOCH_FIELDS)
        w.writeheader()
        for fr in fold_reports:
            for row in fr["epoch_logs"]:
                w.writerow(
                    _flat_metrics_row(
                        row["fold"],
                        row["epoch"],
                        row["val"],
                        row["test"],
                    )
                )

    keys = ["fold", "best_epoch"] + _PER_EPOCH_FIELDS[2:]
    best_rows: list[dict] = []
    for fr in fold_reports:
        b = fr["best_by_val_f1"]
        best_rows.append(
            {
                "fold": fr["fold"],
                "best_epoch": b["epoch"],
                "val_accuracy": b["val"]["accuracy"],
                "val_precision_macro": b["val"]["precision"],
                "val_recall_macro": b["val"]["recall"],
                "val_f1_macro": b["val"]["f1"],
                "test_accuracy": b["test"]["accuracy"],
                "test_precision_macro": b["test"]["precision"],
                "test_recall_macro": b["test"]["recall"],
                "test_f1_macro": b["test"]["f1"],
            }
        )

    mean_row: dict = {"fold": "MEAN", "best_epoch": ""}
    std_row: dict = {"fold": "STD", "best_epoch": ""}
    for k in keys[2:]:
        vals = [row[k] for row in best_rows]
        mean_row[k] = float(np.mean(vals))
        std_row[k] = float(np.std(vals))

    with open(path_best, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=keys)
        w.writeheader()
        for row in best_rows:
            w.writerow(row)
        w.writerow(mean_row)
        w.writerow(std_row)

    def fmt(x: object, w: int) -> str:
        if isinstance(x, float):
            s = f"{x:.4f}"
        else:
            s = str(x)
        return s[:w].ljust(w)

    cw = {"fold": 6, "ep": 6, "m": 8}
    hdr = (
        f"{fmt('fold', cw['fold'])} {fmt('epoch', cw['ep'])} "
        f"{fmt('v_acc', cw['m'])} {fmt('v_Pm', cw['m'])} {fmt('v_Rm', cw['m'])} {fmt('v_Fm', cw['m'])} "
        f"{fmt('t_acc', cw['m'])} {fmt('t_Pm', cw['m'])} {fmt('t_Rm', cw['m'])} {fmt('t_Fm', cw['m'])}"
    )
    sep = "-" * len(hdr)
    lines = [
        "\n" + "=" * 96,
        "RESULTS (3-way; P/R/F1 = macro over classes) — best val macro-F1 per fold",
        "=" * 96,
        hdr,
        sep,
    ]
    for row in best_rows + [mean_row, std_row]:
        lines.append(
            f"{fmt(row['fold'], cw['fold'])} {fmt(row.get('best_epoch', ''), cw['ep'])} "
            f"{fmt(row['val_accuracy'], cw['m'])} {fmt(row['val_precision_macro'], cw['m'])} "
            f"{fmt(row['val_recall_macro'], cw['m'])} {fmt(row['val_f1_macro'], cw['m'])} "
            f"{fmt(row['test_accuracy'], cw['m'])} {fmt(row['test_precision_macro'], cw['m'])} "
            f"{fmt(row['test_recall_macro'], cw['m'])} {fmt(row['test_f1_macro'], cw['m'])}"
        )
    lines.append(sep)
    lines.append(f"Saved:\n  {os.path.abspath(path_epoch)}\n  {os.path.abspath(path_best)}")
    print("\n".join(lines))
    return path_epoch, path_best


def train_fold(
    model: nn.Module,
    train_loader: DataLoader,
    val_eval_loader: DataLoader,
    test_eval_loader: DataLoader,
    device: torch.device,
    fold_idx: int,
) -> dict:
    optimizer = torch.optim.AdamW(model.parameters(), lr=config.lr, weight_decay=config.weight_decay)
    t_total = len(train_loader) * config.epochs
    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=int(0.06 * t_total), num_training_steps=t_total
    )
    use_cuda = device.type == "cuda"
    autocast_ctx, scaler = build_amp_helpers(use_cuda)

    best_val_f1 = -1.0
    best_snapshot: dict | None = None
    epoch_logs: list[dict] = []

    for epoch in range(config.epochs):
        model.train()
        optimizer.zero_grad(set_to_none=True)
        pbar = tqdm(train_loader, desc=f"Fold {fold_idx + 1} | Epoch {epoch + 1}/{config.epochs}")
        for step, batch in enumerate(pbar):
            labels = batch.pop("labels").to(device)
            batch = {k: v.to(device) for k, v in batch.items()}
            with autocast_ctx():
                out = model(**batch, labels=labels)
                loss = out["loss"] / config.grad_accumulation
            if use_cuda:
                scaler.scale(loss).backward()
            else:
                loss.backward()
            if (step + 1) % config.grad_accumulation == 0:
                if use_cuda:
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    optimizer.step()
                scheduler.step()
                optimizer.zero_grad(set_to_none=True)

        y_val, p_val = collect_predictions(model, val_eval_loader, device, autocast_ctx)
        y_te, p_te = collect_predictions(model, test_eval_loader, device, autocast_ctx)
        m_val = multiclass_macro_metrics(y_val, p_val)
        m_test = multiclass_macro_metrics(y_te, p_te)
        epoch_logs.append({"fold": fold_idx + 1, "epoch": epoch + 1, "val": m_val, "test": m_test})

        print(
            f"Fold {fold_idx + 1} | Epoch {epoch + 1}/{config.epochs} | "
            f"VAL  acc={m_val['accuracy']:.4f} P_macro={m_val['precision']:.4f} "
            f"R_macro={m_val['recall']:.4f} F1_macro={m_val['f1']:.4f} | "
            f"TEST acc={m_test['accuracy']:.4f} P_macro={m_test['precision']:.4f} "
            f"R_macro={m_test['recall']:.4f} F1_macro={m_test['f1']:.4f}"
        )

        if m_val["f1"] > best_val_f1:
            best_val_f1 = m_val["f1"]
            best_snapshot = {"epoch": epoch + 1, "val": dict(m_val), "test": dict(m_test)}

    assert best_snapshot is not None
    print(
        f"\n--- Fold {fold_idx + 1} best (by val macro-F1 @ epoch {best_snapshot['epoch']}) ---\n"
        f"  Val  : acc={best_snapshot['val']['accuracy']:.4f} P/R/F1_macro="
        f"{best_snapshot['val']['precision']:.4f}/{best_snapshot['val']['recall']:.4f}/{best_snapshot['val']['f1']:.4f}\n"
        f"  Test : acc={best_snapshot['test']['accuracy']:.4f} P/R/F1_macro="
        f"{best_snapshot['test']['precision']:.4f}/{best_snapshot['test']['recall']:.4f}/{best_snapshot['test']['f1']:.4f}\n"
    )
    return {"fold": fold_idx + 1, "best_by_val_f1": best_snapshot, "epoch_logs": epoch_logs}


def main():
    from datasets import load_dataset

    assert abs(config.train_frac + config.val_frac + config.test_frac - 1.0) < 1e-6

    device = resolve_training_device(force_cpu=config.force_cpu)
    print("Using device:", device)
    print("3-way classes:", THREE_WAY_NAMES)
    print(
        "Metadata:",
        f"use_metadata_prefix={config.use_metadata_prefix}, use_split_flags={config.use_split_flags}",
    )

    print("Loading Fakeddit dataset...")
    ds = load_dataset(config.dataset_name, split="train")
    n = min(config.max_samples, len(ds))
    ds = ds.select(range(n))
    print(f"Total samples: {len(ds)}")

    processor = ViltProcessor.from_pretrained(config.model_name)
    vcfg = AutoConfig.from_pretrained(config.model_name)
    if getattr(vcfg, "image_size", None) is not None:
        config.image_size = int(vcfg.image_size)

    valid_idx = build_valid_indices(ds, len(ds))
    print(f"Valid samples: {len(valid_idx)}/{len(ds)}")

    labels_all = [six_way_to_three_way(int(ds[i]["6_way_label"])) for i in valid_idx]
    y_all = np.array(labels_all, dtype=np.int64)
    counts = np.bincount(y_all, minlength=config.num_classes)
    print("3-way class counts [true, satire, misinfo]:", counts.tolist())

    pos = np.arange(len(valid_idx))
    pos_train, pos_temp, y_tr, y_tmp = train_test_split(
        pos,
        y_all,
        test_size=(config.val_frac + config.test_frac),
        stratify=y_all,
        random_state=SEED,
    )
    rel_test = config.test_frac / (config.val_frac + config.test_frac)
    pos_val, pos_test, _, _ = train_test_split(
        pos_temp,
        y_tmp,
        test_size=rel_test,
        stratify=y_tmp,
        random_state=SEED,
    )

    abs_train_pool = [valid_idx[int(i)] for i in pos_train]
    abs_val = [valid_idx[int(i)] for i in pos_val]
    abs_test = [valid_idx[int(i)] for i in pos_test]
    y_train_pool = y_tr

    print(
        f"\nSplit sizes — train pool: {len(abs_train_pool)} ({100 * len(abs_train_pool) / len(valid_idx):.1f}%), "
        f"val: {len(abs_val)} ({100 * len(abs_val) / len(valid_idx):.1f}%), "
        f"test: {len(abs_test)} ({100 * len(abs_test) / len(valid_idx):.1f}%)\n"
    )

    val_ds = FakedditThreeWayDataset(ds, processor, abs_val)
    test_ds = FakedditThreeWayDataset(ds, processor, abs_test)
    val_loader = DataLoader(
        val_ds,
        batch_size=config.batch_size,
        shuffle=False,
        num_workers=config.num_workers,
        collate_fn=collate_fn,
    )
    test_loader = DataLoader(
        test_ds,
        batch_size=config.batch_size,
        shuffle=False,
        num_workers=config.num_workers,
        collate_fn=collate_fn,
    )

    skf = StratifiedKFold(n_splits=config.n_splits, shuffle=True, random_state=SEED)
    fold_reports: list[dict] = []

    for fold_idx, (tr_rel, _) in enumerate(skf.split(np.zeros(len(abs_train_pool)), y_train_pool)):
        print("\n" + "=" * 72)
        print(f"Fold {fold_idx + 1}/{config.n_splits} (train subset from 80% pool)")
        print("=" * 72)
        fold_train_abs = [abs_train_pool[i] for i in tr_rel]
        print(f"  This-fold train size: {len(fold_train_abs)}")

        train_ds = FakedditThreeWayDataset(ds, processor, fold_train_abs)
        train_loader = DataLoader(
            train_ds,
            batch_size=config.batch_size,
            shuffle=True,
            num_workers=config.num_workers,
            collate_fn=collate_fn,
        )

        model = ViltThreeWayDetector(config.num_classes).to(device)
        report = train_fold(model, train_loader, val_loader, test_loader, device, fold_idx)
        fold_reports.append(report)

    print("\n" + "=" * 72)
    print("SUMMARY (best epoch per fold by val macro-F1)")
    print("=" * 72)
    for fr in fold_reports:
        b = fr["best_by_val_f1"]
        print(
            f"Fold {fr['fold']}: best epoch={b['epoch']} | "
            f"VAL acc={b['val']['accuracy']:.4f} F1m={b['val']['f1']:.4f} | "
            f"TEST acc={b['test']['accuracy']:.4f} F1m={b['test']['f1']:.4f}"
        )

    test_f1s = [fr["best_by_val_f1"]["test"]["f1"] for fr in fold_reports]
    test_acc = [fr["best_by_val_f1"]["test"]["accuracy"] for fr in fold_reports]
    val_f1s = [fr["best_by_val_f1"]["val"]["f1"] for fr in fold_reports]
    print(
        f"\nMean ± std (best-by-val-macro-F1 snapshot):\n"
        f"  Val F1_macro  : {float(np.mean(val_f1s)):.4f} ± {float(np.std(val_f1s)):.4f}\n"
        f"  Test F1_macro : {float(np.mean(test_f1s)):.4f} ± {float(np.std(test_f1s)):.4f}\n"
        f"  Test Acc      : {float(np.mean(test_acc)):.4f} ± {float(np.std(test_acc)):.4f}\n"
    )

    save_and_print_results_tables(fold_reports)


if __name__ == "__main__":
    main()
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

Using device: cuda
3-way classes: ('true', 'satire', 'misinformation')
Metadata: use_metadata_prefix=True, use_split_flags=True
Loading Fakeddit dataset...


README.md:   0%|          | 0.00/696 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/73.9M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/794660 [00:00<?, ? examples/s]

Total samples: 5000


...

/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


Fold 5 | Epoch 4/5 | VAL  acc=0.7637 P_macro=0.8413 R_macro=0.6021 F1_macro=0.6201 | TEST acc=0.7695 P_macro=0.7527 R_macro=0.6184 F1_macro=0.6367


Fold 5 | Epoch 5/5:   0%|          | 0/278 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


Fold 5 | Epoch 5/5 | VAL  acc=0.7666 P_macro=0.7686 R_macro=0.6386 F1_macro=0.6668 | TEST acc=0.7493 P_macro=0.7199 R_macro=0.6092 F1_macro=0.6342

--- Fold 5 best (by val macro-F1 @ epoch 5) ---
  Val  : acc=0.7666 P/R/F1_macro=0.7686/0.6386/0.6668
  Test : acc=0.7493 P/R/F1_macro=0.7199/0.6092/0.6342


SUMMARY (best epoch per fold by val macro-F1)
Fold 1: best epoch=2 | VAL acc=0.7464 F1m=0.6443 | TEST acc=0.7061 F1m=0.5122
Fold 2: best epoch=4 | VAL acc=0.7608 F1m=0.7057 | TEST acc=0.7493 F1m=0.6341
Fold 3: best epoch=3 | VAL acc=0.7579 F1m=0.7267 | TEST acc=0.7637 F1m=0.6542
Fold 4: best epoch=4 | VAL acc=0.7839 F1m=0.7074 | TEST acc=0.7666 F1m=0.6029
Fold 5: best epoch=5 | VAL acc=0.7666 F1m=0.6668 | TEST acc=0.7493 F1m=0.6342

Mean ± std (best-by-val-macro-F1 snapshot):
  Val F1_macro  : 0.6902 ± 0.0301
  Test F1_macro : 0.6075 ± 0.0504
  Test Acc      : 0.7470 ± 0.0217


RESULTS (3-way; P/R/F1 = macro over classes) — best val macro-F1 per fold
fold   epoch  v_acc    v_Pm     v_R